# Class 15: Dynamics of Networks — Temporal Networks

Goal of today's class:
1. Create temporal networks from data
2. Learn key properties of temporal networks like inter-event time distributions, burstiness, etc.
3. Introduce the activity-driven network model

*Acknowledgement: Some of the material in this lesson is based on materials from Matteo Chinazzi and Qian Zhang.*
__________

## Getting today's notebook

**Every class, duplicate the notebook before you run anything, and work in the copy.** Running a cell changes the file, and a changed original can block your next pull.

1. Log in to [Open OnDemand](https://ood.explorer.northeastern.edu/), open the **Courses** menu, and click **JupyterLab (Python)**. Launch it with the same settings as last class; they should already be filled in.
2. Once JupyterLab starts, open a terminal (**File → New → Terminal**) and pull the new material, where `USERNAME` is your Northeastern username:
   ```bash
   cd /courses/NETS7052.202710/students/USERNAME/nets7052_fa26
   git fetch upstream
   git checkout main
   git merge --no-edit upstream/main
   ```
   Once you've committed work of your own to your fork (Assignment 1, for example), this merge needs a commit message. `--no-edit` lets Git write that message itself instead of opening a text editor in your terminal.
3. **Duplicate today's notebook before you run anything.** In the file browser, open `notebooks/class_15_temporal/`, right-click `class_15_temporal.ipynb`, and choose **Duplicate**. Open the copy (`class_15_temporal-Copy1.ipynb`) and work there.
4. Check that the top-right corner of the notebook says **Python (cmns-core)**. If it doesn't, choose **Kernel → Change Kernel…** and pick it.

**If the merge doesn't work:**

- *"Your local changes to the following files would be overwritten by merge"*: you changed a notebook that I've since updated, and Git lists which one. If you took notes in it, duplicate it first. Then close it and reset it, using the path Git listed:
  ```bash
  git restore notebooks/class_14_dynamics3/class_14_dynamics3.ipynb
  ```

  `git restore` throws away every change you made to that one file and puts back the version from your last pull. Git can't undo it, which is why you duplicate first. Then run `git merge --no-edit upstream/main` again.
- *"'upstream' does not appear to be a git repository"*: you missed a step in Class 0. Run this, then start again from `git fetch upstream`:
  ```bash
  git remote add upstream https://github.com/computational-methods-network-science/nets7052_fa26.git
  ```
- **Git asks for a GitHub username or password:** don't type them. Your `upstream` address is wrong. Run this, then start again from `git fetch upstream`:
  ```bash
  git remote set-url upstream https://github.com/computational-methods-network-science/nets7052_fa26.git
  ```

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rc
rc('axes', axisbelow=True)
rc('axes', fc='w')
rc('figure', fc='w')
rc('savefig', fc='w')

# folders for the figures we save today
import os
os.makedirs('images/pngs', exist_ok=True)
os.makedirs('images/pdfs', exist_ok=True)

________
**(Pre) pre-class challenge!!!**

1. Create a barabasi-albert graph (N=10000, m=2)
2. For T iterations, randomly rewire edges in the graph
3. At each iteration, calculate the standard deviation of the degree distribution.
4. Plot over time.

In [ ]:
import random
import statistics


N = 10000; m = 2
G = nx.barabasi_albert_graph(N, m)
T = 100

t_ = 0
for t_ in range(T):
    # pick random node
    # pick random edge
    first_node = random.randint(1,len(list(G.nodes)))
    edges = list(G.edges(first_node))
    rand_edge = random.randint(1,len(edges)-1)
    print(edges[rand_edge])
    G.remove_edge(edges[rand_edge][0], edges[rand_edge][1])

    second_node = random.randint(1,len(list(G.nodes)))
    print(f"new edge: {(first_node, second_node)}")
    G.add_edge(first_node, second_node)

    

    t_ += 1
    
    print(statistics.stdev(list(dict(G.degree()).values())))
    

In [ ]:
pass

_________
(don't look below!)

In [ ]:
# N = 1000 here, not 10000: T is then about 5 times the number of edges, enough to see the curve flatten
G = nx.barabasi_albert_graph(1000, 2)
T = 10000

degs = np.array(list(dict(G.degree()).values()))
k2s = [np.std(degs)]

In [ ]:
for _ in range(T):
    # pick a random edge
    edges = list(G.edges())
    edge_inds = list(range(len(edges)))
    nodes = set(G.nodes())
    rand_eij = edges[np.random.choice(edge_inds)]

    # rewire one of the ends to another random node
    keep_i = rand_eij[np.random.choice([0,1])]
    eligible_j = nodes - {keep_i} - set(G.neighbors(keep_i))
    new_j = np.random.choice(list(eligible_j))

    G.remove_edge(rand_eij[0],rand_eij[1])
    G.add_edge(keep_i, new_j)

    degs = list(dict(G.degree()).values())
    k2s.append(np.std(degs))


In [ ]:
fig, ax = plt.subplots(1,1,figsize=(6,3),dpi=100)

ax.plot(k2s, lw=2, color='.2')
ax.set_xlabel('Number of rewires')
ax.set_ylabel('Standard deviation of degrees')

ax.set_xlim(-100, T+100)

plt.show()

________
(for real this time)
### To begin: a challenge.

In the `data` folder, there is a file `sp.tsv`.
1. Use `pandas` to load this dataset.
2. Name the columns ['timestamp', 'node_i', 'node_j']
3. Plot the number of connections between pairs of nodes per unit time.

**You have 10 minutes to create a figure.**

In [ ]:
pass

____
## Temporal Networks (Time-Varying Networks)

To motivate, what does *contact tracing* actually look like in an epidemic?

![](images/contact_tracing.png)

_________

## Temporal networks are everywhere and also every freakin' thing

(Controversial claim?) **Nearly every system we study is better described as a sequence of interactions in time rather than a static web of ties.**

What matters is not only who is connected to whom, but when and in what order events occur—because the ordering of contacts constrains causality: information, infection, and influence can only flow along time-respecting paths. This perspective unifies settings that look very different on the surface—communication, proximity, biology, infrastructure, brains, and ecology—yet share recurring temporal signatures such as burstiness, diurnal rhythms, and nonstationarity (e.g., Holme & Saramäki, 2012; Masuda & Lambiotte, 2016).


Across all of these domains, three methodological themes recur: (i) represent interactions as contact sequences or *time-unfolded* graphs so that ordering is explicit; (ii) choose analysis windows deliberately, because window size can flip conclusions; and (iii) benchmark against temporal nulls (time-shuffle, endpoint-shuffle, inter-event-preserving) to isolate what is genuinely temporal from what is a by-product of sampling. The result is a notion of structure that is causal, not just topological...and that’s the point of studying temporal networks at all!



#### 1) Person-to-person communication (email, calls/messages, online DMs)

* Events as $(u,v,t,\Delta t)$; ordering constrains who can reach whom.
* Bursty, correlated gaps → earliest-arrival/fastest paths matter.
* Temporal centrality (foremost/fastest) can invert static rankings.
* Window choice (1h vs 24h) changes reachability and “influence.”


#### 2) One-to-many communication (Twitter, Instagram, Wikipedia)

* Broadcasts form cascades whose shape depends on timing, not just topology.
* Diurnal/session rhythms gate who can plausibly influence whom.
* Temporal trees (time-respecting) differ from static diffusion assumptions.
* Timestamp shuffles separate temporal effects from follower/editor structure.


#### 3) Physical proximity (Reality Mining, SocioPatterns)

* Minute-scale contact sequences with strong daily/weekly cycles.
* Static aggregation invents paths; time-respecting reachability is smaller/slower.
* Temporal motifs (ordered triads) reveal handoffs across classes/sessions.
* Superspreaders can be session-specific, not globally high-degree.


#### 4) Infrastructural systems (air, train, bus; GTFS/timetables)

* Edges exist only at scheduled times; feasibility = ordering-consistent routes.
* Optimize earliest-arrival/fastest, not topological shortest path.
* Rush-hour centrality/bottlenecks are time-localized.
* Delays = timestamp perturbations; robustness is inherently temporal.


#### 5) Biological & ecological dynamics (PPI, GRNs, metabolism; food webs, animal mobility)

* Edges switch with state/season; viable paths are transient.
* Direction/irreversibility require time-ordered paths (causal feasibility).
* Temporal multilayers recover control points and seasonal bottlenecks.
* Aggregation masks phase-specific articulation nodes and bridges.


____
### Time is relative!
* $t_G$ describes the evolution of the network
* $t_P$ describes the evolution of the process
* scale of $t_G/t_P$: if the network changes much faster than the process, the annealed approximation works; if it changes much slower, the static approximation works; if the two are comparable, you need the temporal network itself

<!-- ![](images/timescale.png) -->

### Representation
* Static network: adjacency matrix $A_{ij}$

* Temporal network: adjacency matrix as a function of time $A_{ij}(t)$


<!-- ### Shortest path

![](images/shortestpath1.png)
__________
 -->

<!-- ![](images/shortestpath2.png)
__________
 -->

<!-- ![](images/shortestpath3.png)
__________
 -->

<!-- ![](images/shortestpath4.png)
__________ -->

### Connected components

I. Strongly connected: all $i$ and $j$ reachable within $T$ (in both directions)

II. Weakly connected: as above, ignoring the direction of contacts (on undirected contact data like ours, it's the same as strongly connected)

### Centralities

I. Closeness:

$$ c^{c}(i) = \frac{N-1}{\sum_{i\neq j} d_{ij}} \qquad \Rightarrow \qquad c^{c}(i, T) = \frac{N-1}{\sum_{i\neq j} \lambda_{ijT}} $$

where $\lambda_{ijT}$ is the temporal distance from $i$ to $j$ within the window $T$: the time it takes along the fastest time-respecting path. If any node can't be reached, that sum is infinite and $c^c(i,T)=0$, so temporal networks usually use the reciprocal form instead: $c^{e}(i,T)=\frac{1}{N-1}\sum_{j\neq i}\frac{1}{\lambda_{ijT}}$, with $1/\lambda_{ijT}=0$ when there's no path.

II. Betweenness:
* $g_{j,k}$ is the number of shortest paths between nodes $j$ and $k$
* $g_{j,k}(i)$ is the number of shortest paths between nodes $j$ and $k$ going through $i$ 
* if $j=k$, let $g_{jk} = 1$, and if $i\in\{j,k\}$, let $g_{j,k}(i)=0$; a pair with no path between them contributes 0

$$ c^b(i) = \sum \frac{g_{j,k}(i)}{g_{j,k}} \qquad \Rightarrow \qquad c^b(i, T) = \sum \frac{g_{j,k}(i, T)}{g_{j,k}(T)} $$


______
### Loading temporal network data

Revisiting the temporal network from above...

The **High School Contact Network Dataset** from the [SocioPatterns](https://www.sociopatterns.org/) project captures the temporal interactions between students in a French high school over the course of several days. The dataset was collected using wearable proximity sensors that recorded face-to-face interactions among students at 20-second intervals. Each interaction includes the IDs of the two individuals involved and the timestamp of the interaction in seconds since the start of the observation period.

We're using the 2012 collection: 180 students in five classes at a high school in Marseilles, over seven school days in November 2012 (Monday through the following Tuesday, so look for the weekend gap). SocioPatterns shares it under a CC BY-NC-SA 3.0 license and asks that you cite Fournet & Barrat (2014).

This dataset is widely used in network science research to study temporal networks, social interactions, and spreading processes (e.g., information or disease propagation). Its temporal resolution and rich interaction data make it an excellent resource for exploring dynamic graph structures and understanding social behavior in structured environments like schools.

**References**
- Fournet, J., & Barrat, A. (2014). Contact patterns among high school students. *PLoS ONE*, 9(9), e107878. https://doi.org/10.1371/journal.pone.0107878
- Cattuto, C., Van den Broeck, W., Barrat, A., Colizza, V., Pinton, J. F., & Vespignani, A. (2010). Dynamics of person-to-person interactions from distributed RFID sensor networks. *PloS One*, 5(7), e11596. https://doi.org/10.1371/journal.pone.0011596
- Starnini, M., Baronchelli, A., & Pastor-Satorras, R. (2013). Modeling human dynamics of face-to-face interaction networks. *Physical Review Letters*, 110(16), 168701. https://doi.org/10.1103/PhysRevLett.110.168701

In [ ]:
# time \t node_i \t node_j
tdf = pd.read_csv('data/sp.tsv',sep='\t',header=None,
                  names=['timestamp','node_i','node_j'])[['node_i','node_j','timestamp']]

In [ ]:
bin_size = 60*1 # 1 minute
tdf["time_bin"] = (tdf['timestamp'] // bin_size) * bin_size

In [ ]:
tdf

In [ ]:
aggregate_counts = tdf["time_bin"].value_counts().sort_index()
aggregate_counts

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(9,3),dpi=200)

ax.bar(aggregate_counts.index, aggregate_counts.values, width=bin_size, color='.2', alpha=0.8)

ax.grid(axis="y", linestyle="--", alpha=0.5)
# Set x-ticks at absolute 24-hour intervals
start_time = aggregate_counts.index.min()
end_time = aggregate_counts.index.max()

# Generate 24-hour spaced timestamps
tick_positions = range(start_time, end_time + 1, 24 * 3600)  # Step by 24 hours (in seconds)
tick_labels = ['Day %i'%i for i in range(1,len(tick_positions)+1)]

# Set ticks and labels
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)

ax.set_title("Connections observed at %.3f-hour increments"%(bin_size/(3600)))
ax.set_xlabel("Time")
ax.set_ylabel("Contact records (20 s each)")


plt.show()

In [ ]:
bin_size = 60*60 # 1 hour

tdf["time_bin"] = (tdf['timestamp'] // bin_size) * bin_size

aggregate_counts = tdf["time_bin"].value_counts().sort_index()

fig, ax = plt.subplots(1,1,figsize=(9,3),dpi=200)

ax.bar(aggregate_counts.index, aggregate_counts.values,
       width=bin_size, color='steelblue', alpha=0.8, ec='.2', lw=0.25)

ax.grid(axis="y", linestyle="--", alpha=0.5)
# Set x-ticks at absolute 24-hour intervals
start_time = aggregate_counts.index.min()
end_time = aggregate_counts.index.max()

# Generate 24-hour spaced timestamps
tick_positions = range(start_time, end_time + 1, 24 * 3600)  # Step by 24 hours (in seconds)
tick_labels = ['Day %i'%i for i in range(1,len(tick_positions)+1)]

# Set ticks and labels
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)

ax.set_title("Connections observed at %.1f-hour increments"%(bin_size/(3600)))
ax.set_xlabel("Time")
ax.set_ylabel("Contact records (20 s each)")


plt.show()

In [ ]:
bin_size = 60*60*24 # 24 hour

tdf["time_bin"] = (tdf['timestamp'] // bin_size) * bin_size

aggregate_counts = tdf["time_bin"].value_counts().sort_index()

fig, ax = plt.subplots(1,1,figsize=(9,3),dpi=200)

ax.bar(aggregate_counts.index, aggregate_counts.values, width=bin_size,
       color='palevioletred', alpha=0.8, ec='.2')

ax.grid(axis="y", linestyle="--", alpha=0.5)
# Set x-ticks at absolute 24-hour intervals
start_time = aggregate_counts.index.min()
end_time = aggregate_counts.index.max()

# Generate 24-hour spaced timestamps
tick_positions = range(start_time, end_time + 1, 24 * 3600)  # Step by 24 hours (in seconds)
tick_labels = ['Day %i'%i for i in range(1,len(tick_positions)+1)]

# Set ticks and labels
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)

ax.set_title("Connections observed at %.1f-hour increments"%(bin_size/(3600)))
ax.set_xlabel("Time")
ax.set_ylabel("Contact records (20 s each)")


plt.show()

____
### But what about the network!

In [ ]:
def temporal_network_from_data(
    data,
    aggregation="all",
    interval_size=1,
    time_window=None,
    custom_slices=None,
    directed=False,
    multigraph=False,
    column_names=None
):
    """
    Convert temporal edge data into NetworkX graph objects with flexible temporal aggregation.

    This function processes temporal edge data (provided as a DataFrame, CSV file, or NumPy array)
    to create NetworkX graphs. It supports various temporal aggregation methods, including grouping
    by fixed intervals, custom time slices, or a single aggregated graph for all data.

    Parameters:
    ----------
    data : pd.DataFrame, str, or np.ndarray
        Input temporal edge data. Can be:
        - A Pandas DataFrame with required columns: 'node_i', 'node_j', 'timestamp'.
        - A file path to a CSV file (requires `column_names` to be specified).
        - A NumPy array with columns matching the required structure (requires `column_names`).

    aggregation : str, optional
        Temporal aggregation method. Options are:
        - 'interval': Group data into fixed time intervals (specified by `interval_size`).
        - 'custom': Group data based on custom time slices (specified by `custom_slices`).
        - 'all': Aggregate all data into a single graph.
        Default is 'all'.

    interval_size : int, optional
        Size of the time intervals (in seconds) for 'interval' aggregation. Default is 1.

    time_window : tuple, optional
        A tuple `(start, stop)` to filter timestamps within a specific range (in seconds).
        Default is None, which includes all timestamps.

    custom_slices : list, optional
        List of custom slice sizes (in seconds) for 'custom' aggregation. For example, [3600, 7200]
        would create two time bins of 1 hour and 2 hours, respectively. Default is None.

    directed : bool, optional
        If True, creates directed graphs (DiGraph or MultiDiGraph). Default is False.

    multigraph : bool, optional
        If True, creates MultiGraphs or MultiDiGraphs to allow parallel edges. If False,
        standard Graph or DiGraph objects are created, and weights are aggregated. Default is False.

    column_names : list, optional
        Required when `data` is a CSV file or NumPy array. Specifies the column names as:
        ['node_i', 'node_j', 'timestamp', (optional) 'weight'].

    Returns:
    -------
    dict or networkx.Graph
        - If data is aggregated into multiple time bins (e.g., 'interval' or 'custom'), returns a
          dictionary where keys are time bins and values are NetworkX graph objects.
        - If data is aggregated into a single graph (e.g., 'all'), returns a single NetworkX graph.

    Raises:
    ------
    ValueError
        - If required columns ('node_i', 'node_j', 'timestamp') are missing.
        - If `custom_slices` is not provided for 'custom' aggregation.
        - If `column_names` is not provided when input data is a CSV file or NumPy array.

    TypeError
        - If input data is not a Pandas DataFrame, a CSV file path, or a NumPy array.

    Examples:
    --------
    1. Aggregating into a single graph:
        >>> G_agg = temporal_network_from_data(data, aggregation="all")

    2. Aggregating into 1-hour intervals:
        >>> G_interval = temporal_network_from_data(data, aggregation="interval", interval_size=3600)

    3. Using custom time slices:
        >>> G_custom = temporal_network_from_data(data, aggregation="custom", custom_slices=[3600, 7200])

    4. Handling a CSV file:
        >>> G_csv = temporal_network_from_data("edges.csv",
                                               column_names=["node_i","node_j","timestamp","weight"])

    Notes:
    -----
    - The 'weight' column, if present, is aggregated for standard graphs (not MultiGraphs).
    - The timestamp column must be numeric and represents time in seconds.
    """

    # Handle data input
    if isinstance(data, pd.DataFrame):
        df = data.copy()  # don't overwrite the caller's columns
    elif isinstance(data, str):  # CSV file path
        if column_names is None:
            raise ValueError("Column names must be provided when reading from a CSV file.")
        df = pd.read_csv(data, names=column_names)
    elif isinstance(data, np.ndarray):  # NumPy array
        if column_names is None:
            raise ValueError("Column names must be provided for a NumPy array.")
        df = pd.DataFrame(data, columns=column_names)
    else:
        raise TypeError("Input data must be a Pandas DataFrame, a CSV file path, or a NumPy array.")
    
    # Check required columns
    required_columns = {"node_i", "node_j", "timestamp"}
    if not required_columns.issubset(df.columns):
        raise ValueError(f"Input data must contain {required_columns} columns.")
    
    # On an undirected graph, (i, j) and (j, i) are the same edge, so put each pair in one order
    if not directed:
        swap = df["node_i"] > df["node_j"]
        df.loc[swap, ["node_i", "node_j"]] = df.loc[swap, ["node_j", "node_i"]].values

    # Filter by time window
    if time_window:
        start, stop = time_window
        df = df[(df["timestamp"] >= start) & (df["timestamp"] <= stop)].copy()
    
    # Determine time bins
    if aggregation == "interval":
        df["time_bin"] = (df["timestamp"] // interval_size) * interval_size
    elif aggregation == "custom":
        if not custom_slices:
            raise ValueError("Custom slices must be provided for 'custom' aggregation.")
        time_bins = []
        current_start = df["timestamp"].min()
        for slice_size in custom_slices:
            current_end = current_start + slice_size
            time_bins.append((current_start, current_end))
            current_start = current_end
        
        # Assign rows to custom time bins
        def assign_to_custom_bin(ts):
            for i, (start, end) in enumerate(time_bins):
                if start <= ts < end:
                    return i
            return None  # Outside custom slices
        
        df["time_bin"] = df["timestamp"].apply(assign_to_custom_bin)
        df = df[df["time_bin"].notnull()]  # Remove rows outside custom slices
    elif aggregation == "all":
        df["time_bin"] = "all"  # Single graph for all data
    else:
        raise ValueError(f"Unsupported aggregation type: {aggregation}")
    
    # Group by time bins and create graphs
    graphs = {}
    for time_bin, group in df.groupby("time_bin"):
        # Determine graph type
        if multigraph:
            G = nx.MultiDiGraph() if directed else nx.MultiGraph()
        else:
            G = nx.DiGraph() if directed else nx.Graph()
        
        # Aggregate weights if multigraph is False
        if not multigraph:
            edge_weights = (
                group.groupby(["node_i", "node_j"])
                .agg(weight=("weight", "sum") if "weight" in group.columns else ("timestamp", "size"))
                .reset_index()
            )
            for _, row in edge_weights.iterrows():
                G.add_edge(row["node_i"], row["node_j"], weight=row["weight"])
        else:  # Add edges directly without aggregation
            for _, row in group.iterrows():
                G.add_edge(
                    row["node_i"], 
                    row["node_j"], 
                    weight=row.get("weight", 1), 
                    timestamp=row["timestamp"]
                )
        
        graphs[time_bin] = G
    
    # Return a single graph if no aggregation
    if aggregation == "all" or (aggregation == "custom" and len(graphs) == 1):
        return next(iter(graphs.values()))


    return graphs

Test out our temporal network creation function! Visualize the **aggregated** network.

In [ ]:
G_agg = temporal_network_from_data(tdf, aggregation="all")

In [ ]:
np.random.seed(5)
pos = nx.spring_layout(G_agg, k=1.0, iterations=200, threshold=0.005)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7,6), dpi=100)

# Extract edge weights
ews = np.array(list(nx.get_edge_attributes(G_agg, 'weight').values()))
ews = ews ** 0.3 - 0.5 # Scale edge width non-linearly for visualization

# Relate edge colors to the normalized edge width and adjust with an exponent
ecs = (ews / max(ews)) ** 0.15

# Calculate eigenvector centrality for nodes, adjust with an exponent for better visualization
ncs = np.array(list(nx.eigenvector_centrality(G_agg, weight='weight').values())) ** 0.1
ncs = (ncs-min(ncs))/(max(ncs)-min(ncs))


# Draw the nodes of the network
nx.draw_networkx_nodes(G_agg,pos, node_color=ncs, node_size=40, linewidths=0.5,
                       edgecolors='.1', cmap=plt.cm.magma_r, vmin=0, vmax=1.1, ax=ax)

nx.draw_networkx_edges(G_agg, pos, width=ews, edge_color=ecs, edge_cmap=plt.cm.Greys,
                       edge_vmin=0.45, edge_vmax=1.0, alpha=0.7, ax=ax)


ax.set_title('Time-Aggregated Temporal Network')
ax.set_axis_off()

plt.savefig('images/pngs/aggregated_sp.png',dpi=425, bbox_inches='tight')
plt.savefig('images/pdfs/aggregated_sp.pdf', bbox_inches='tight')

plt.show()

What about if we want a snapshot of the temporal network? E.g. the first hour?

In [ ]:
day = 0
int_size = 60*60
G_t = temporal_network_from_data(tdf, 
                                 aggregation="interval",  # Options: 'interval', 'custom', 'all'
                                 interval_size=int_size, # Size of the time interval for 'interval' aggregation
                                 time_window=None,  # Tuple (start, stop) to restrict aggregation to specific time range
                                 custom_slices=None,  # List of custom slice sizes for aggregation
                                 directed=False,  # Create directed graph
                                 multigraph=False,  # Create a MultiGraph or MultiDiGraph instead of Graph/DiGraph
                                 column_names=None)  # Required if `data` is a CSV or NumPy array

In [ ]:
# G_t

In [ ]:
import itertools as it
nrows = 2
ncols = 5
hours_to_plot = nrows*ncols

fig, ax = plt.subplots(nrows, ncols, figsize=(5*ncols,5*nrows), dpi=200)
tups = list(it.product(range(nrows), range(ncols)))
plt.subplots_adjust(wspace=0.05, hspace=0.05)


for ti, t in enumerate(list(G_t.keys())[:hours_to_plot]):

    a = tups[ti]
    Gt = G_t[t]

    nx.draw_networkx_nodes(G_agg, pos, node_color='.9',
                           node_size=20, linewidths=0.5, edgecolors='.7', ax=ax[a])

    ews = np.array(list(nx.get_edge_attributes(Gt, 'weight').values()))
    ews = ews ** 0.3 - 0.5 # Scale edge width non-linearly for visualization
    ecs = (ews / max(ews)) ** 0.15
    ncs = np.array(list(nx.degree_centrality(Gt).values())) ** 0.1
    ncs = (ncs-min(ncs))/(max(ncs)-min(ncs))

    nx.draw_networkx_nodes(Gt, pos, node_color=ncs, node_size=40, linewidths=0.5,
                           edgecolors='.1', cmap=plt.cm.magma_r, vmin=0, vmax=1.1, ax=ax[a])
    nx.draw_networkx_edges(Gt, pos, width=ews, edge_color=ecs, edge_cmap=plt.cm.Greys,
                           edge_vmin=0.45, edge_vmax=1.0, alpha=0.7, ax=ax[a])

    ax[a].set_title('Hour: %i'%(ti+1), ha='left', va='top', x=0.01,
                    fontweight='bold', y=0.97, color='.35')

plt.suptitle('Temporal Network of High School Students, Day %i'%(day+1), y=0.925, fontsize='xx-large')

plt.show()

In [ ]:
bin_size = 60*60 # 1 hour

tdf["time_bin"] = (tdf['timestamp'] // bin_size) * bin_size

aggregate_counts = tdf["time_bin"].value_counts().sort_index()

fig, ax = plt.subplots(1,1,figsize=(9,3),dpi=200)

ax.bar(aggregate_counts.index, aggregate_counts.values,
       width=bin_size, color='steelblue', alpha=0.8, ec='.2', lw=0.25)

ax.vlines(-60*60, 0, 2000, color='goldenrod')
ax.hlines(2000, -60*60, 60*60*11.5, color='goldenrod')
ax.vlines(60*60*11.5, 0, 2000, color='goldenrod')

ax.grid(axis="y", linestyle="--", alpha=0.5)
# Set x-ticks at absolute 24-hour intervals
start_time = aggregate_counts.index.min()
end_time = aggregate_counts.index.max()

# Generate 24-hour spaced timestamps
tick_positions = range(start_time, end_time + 1, 24 * 3600)  # Step by 24 hours (in seconds)
tick_labels = ['Day %i'%i for i in range(1,len(tick_positions)+1)]

# Set ticks and labels
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)

ax.set_title("Connections observed at %.1f-hour increments"%(bin_size/(3600)))
ax.set_xlabel("Time")
ax.set_ylabel("Contact records (20 s each)")


plt.show()

_______

In [ ]:
def get_binning(data, num_bins=50, is_pmf=False, log_binning=False, threshold=0):
    """
    Bins the input data and calculates the probability mass function (PMF) or 
    probability density function (PDF) over the bins. Supports both linear and 
    logarithmic binning.

    Parameters
    ----------
    data : array-like
        The data to be binned, typically a list or numpy array of values.
    num_bins : int, optional
        The number of bins to use for binning the data (default is 50).
    is_pmf : bool, optional
        If True, computes the probability mass function (PMF) by normalizing 
        histogram counts to sum to 1. If False, computes the probability density 
        function (PDF) by normalizing the density of the bins (default is False).
    log_binning : bool, optional
        If True, uses logarithmic binning with log-spaced bins. If False, uses 
        linear binning (default is False).
    threshold : float, optional
        Only values greater than `threshold` will be included in the binning, 
        allowing for the removal of isolated nodes or outliers (default is 0).
    
    Returns
    -------
    x : numpy.ndarray
        The bin centers, adjusted to be the midpoint of each bin.
    p : numpy.ndarray
        The computed PMF or PDF values for each bin.
    
    Notes
    -----
    This function removes values below a specified threshold, then defines 
    bin edges based on the specified binning method (linear or logarithmic). 
    It calculates either the PMF or PDF based on `is_pmf`.
    """
    
    # Filter out isolated nodes or low values by removing data below threshold
    values = list(filter(lambda x: x > threshold, data))
    if len(values) != len(data):
        print("%s isolated nodes have been removed" % (len(data) - len(values)))

    # Define the range for binning (support of the distribution)
    lower_bound = min(values)
    upper_bound = max(values)

    # Define bin edges based on binning type (logarithmic or linear)
    if log_binning:
        # Use log-spaced bins by taking the log of the bounds
        lower_bound = np.log10(lower_bound)
        upper_bound = np.log10(upper_bound)
        bin_edges = np.logspace(lower_bound, upper_bound, num_bins + 1, base=10)
        # Rounding can nudge the end edges inward, which would silently drop the smallest and largest values
        bin_edges[0], bin_edges[-1] = min(values), max(values)
    else:
        # Use linearly spaced bins
        bin_edges = np.linspace(lower_bound, upper_bound, num_bins + 1)

    # Calculate histogram based on chosen binning method
    if is_pmf:
        # Calculate PMF: normalized counts of data in each bin
        y, _ = np.histogram(values, bins=bin_edges, density=False)
        p = y / y.sum()  # Normalize to get probabilities
    else:
        # Calculate PDF: normalized density of data in each bin
        p, _ = np.histogram(values, bins=bin_edges, density=True)

    # Compute bin centers (midpoints) to represent each bin
    x = bin_edges[1:] - np.diff(bin_edges) / 2  # Bin centers for plotting

    # Remove bins with zero probability to avoid plotting/display issues
    x = x[p > 0]
    p = p[p > 0]

    return x, p

In [ ]:
degrees = np.array(list(dict(G_agg.degree()).values()))
x_agg, y_agg = get_binning(degrees, num_bins=20)

fig, ax = plt.subplots(1,1,figsize=(7.5,4),dpi=100)

ax.loglog(x_agg, y_agg, 'o', color='.3', label='Aggregate network', alpha=0.8, mec='.1')

ax.set_xlabel(r"$k$",fontsize='large')
# plain numbers: the default labels overlap over this short a range
ax.xaxis.set_major_formatter(plt.ScalarFormatter())
ax.xaxis.set_minor_formatter(plt.ScalarFormatter())
ax.set_ylabel(r"$P(k)$",fontsize='large')
ax.legend(fontsize='small')
ax.grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
ax.set_title('SocioPatterns Degree Distribution')

plt.show()

In [ ]:
interval_sizes = [86400, 172800, 259200, 345600, 432000, 518400, 604800, 691200, 777600]
int_s = interval_sizes[0]
G_t = temporal_network_from_data(tdf,
                                 aggregation="interval", # Options: 'interval', 'custom', 'all'
                                 interval_size=int_s, # Size of the time interval for 'interval' aggregation
                                 time_window=None,  # Tuple (start, stop) to restrict aggregation to specific time range
                                 custom_slices=None,  # List of custom slice sizes for aggregation
                                 directed=False,  # Create directed graph
                                 multigraph=False,  # Create a MultiGraph or MultiDiGraph instead of Graph/DiGraph
                                 column_names=None)  # Required if `data` is a CSV or NumPy array

Gt_i = G_t[0]

In [ ]:
degrees = np.array(list(dict(G_agg.degree()).values()))
x_agg, y_agg = get_binning(degrees, num_bins=20)

weights_agg = np.array(list(nx.get_edge_attributes(G_agg, 'weight').values()))
xw_agg, yw_agg = get_binning(weights_agg, num_bins=20)

fig, ax = plt.subplots(1,2,figsize=(12,4),dpi=100)


interval_sizes = [86400, 172800, 259200, 345600, 432000, 518400, 604800, 691200, 777600]
cols = plt.cm.viridis(np.linspace(0,1,1+len(interval_sizes)))[1:]
colsw = plt.cm.magma(np.linspace(0,1,1+len(interval_sizes)))[:-1]

for i,int_s in enumerate(interval_sizes):
    Gt_i = temporal_network_from_data(tdf, aggregation="interval", interval_size=int_s)[0]    
    degrees_ti = np.array(list(dict(Gt_i.degree()).values()))
    x_ti, y_ti = get_binning(degrees_ti, num_bins=20)
    ax[0].loglog(x_ti, y_ti, '-', color=cols[i], label='up to day %i'%(i+1),
              alpha=0.6)
    weights_ti = np.array(list(nx.get_edge_attributes(Gt_i, 'weight').values()))
    xw_ti, yw_ti = get_binning(weights_ti, num_bins=20)
    ax[1].loglog(xw_ti, yw_ti, '-', color=colsw[i], label='up to day %i'%(i+1),
              alpha=0.6)
    
    
ax[0].loglog(x_agg, y_agg, '.-', color='.1', label='Aggregate', mec='.0', ms=10)
ax[1].loglog(xw_agg, yw_agg, '.-', color='.1', label='Aggregate', mec='.0', ms=10)

ax[0].set_xlabel(r"$k$",fontsize='large')
# plain numbers: the default labels overlap over this short a range
ax[0].xaxis.set_major_formatter(plt.ScalarFormatter())
ax[0].xaxis.set_minor_formatter(plt.ScalarFormatter())
ax[0].set_ylabel(r"$P(k)$",fontsize='large')
ax[0].legend(fontsize='x-small',loc=2)
ax[0].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
ax[0].set_title('SocioPatterns Degree Distribution')
ax[1].set_xlabel(r"$w$",fontsize='large')
ax[1].set_ylabel(r"$P(w)$",fontsize='large')
ax[1].legend(fontsize='x-small',loc=1)
ax[1].grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
ax[1].set_title('SocioPatterns Weights Distribution')


plt.show()

In [ ]:
# tmp_times = (tdf['timestamp'].values).copy()
# np.random.shuffle(tmp_times)
# tdf['timestamp_shuffled'] = tmp_times

In [ ]:
t_net = temporal_network_from_data(tdf, aggregation='interval')

In [ ]:
nodes_events = dict()
for t in sorted(t_net.keys()):  # Iterate over sorted time bins
    graph = t_net[t]
    for node in graph.nodes():
        nodes_events.setdefault(node, []).append(t)  # Track event times per node

# Step 2: Calculate inter-event times for each node
event_ie_t = dict()
for node, event_list in nodes_events.items():  # Iterate over nodes and their event times
    for i in range(1, len(event_list)):
        ie_t = event_list[i] - event_list[i - 1]  # Calculate inter-event time
        if ie_t > 20:  # 20 s is the sensor tick: the same contact continuing, not a new event
            event_ie_t.setdefault(event_list[i - 1], []).append(ie_t)  # Group by start time

# Step 3: Flatten the inter-event times for plotting
event_number = []
ie_time_list = []
counter = 0
for t in sorted(event_ie_t.keys()):  # Iterate over sorted event start times
    for ie_t in event_ie_t[t]:
        event_number.append(counter)  # Track event number
        ie_time_list.append(ie_t)  # Add inter-event time
        counter += 1

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(12,5),dpi=200)

ax.plot(event_number, ie_time_list, color='.2', lw=1)

ax.set_xlabel("Event Number")
ax.set_ylabel("Gap between a student's contacts, $t_{ie}$ (s)")
ax.set_title("Inter-Event Time for Each Event")
ax.grid(alpha=0.5)
ax.set_ylim(0)
# ax.set_xlim(0, 80000)

plt.show()

In [ ]:
x_ie, y_ie = get_binning(ie_time_list, num_bins=15, log_binning=True)

fig, ax = plt.subplots(1,1,figsize=(7.5,4),dpi=100)

ax.loglog(x_ie, y_ie, '.-', color='.4', mec='.1', ms=10)

ax.set_xlabel(r"$t_{ie}$ (s)",fontsize='large')
ax.set_ylabel(r"$P(t_{ie})$",fontsize='large')
ax.grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
ax.set_title('SocioPatterns Inter-event Time Distribution')

plt.show()

### Your turn!

Select two global network properties to analyze over time. Pick a timescale to view your network evolution, and create a plot with two subplots that track this property over time.

In [ ]:
# temporal_network_from_data()

In [ ]:
# fig, ax = plt.subplots(1,2,figsize=(9,3),dpi=100)

# plt.show()

In [ ]:
# temporal_network_from_data()

In [ ]:
pass

_______
## Burstiness in Temporal Networks

Temporal networks are systems where interactions or connections between entities occur at specific times. Unlike static networks, temporal networks account for the timing and sequence of events, making them essential for understanding processes such as information diffusion, disease transmission, and human communication patterns.

One intriguing property of temporal networks is **burstiness**---the tendency of events to cluster in short periods of high activity, followed by long periods of inactivity. This phenomenon is observed across various domains, from human interactions (e.g., email exchanges, phone calls) to natural systems (e.g., neuronal firing, animal behavior).

Burstiness arises when the timing of events deviates from a uniform or random (Poissonian) distribution. In a random process, inter-event times (the time gaps between consecutive events) are typically distributed exponentially. However, real-world systems often exhibit heavy-tailed distributions, where a significant fraction of inter-event times are very short, interspersed with much longer intervals.

This non-random clustering of events has significant implications:
- **Efficiency**: Burstiness can accelerate or slow down dynamic processes like spreading phenomena.
- **Predictability**: It reflects underlying mechanisms governing the system, such as behavioral patterns or network constraints.
- **System Design**: Understanding burstiness helps optimize systems, from communication networks to epidemic models.
_________

#### Burstiness Index
The **burstiness index** is a quantitative measure introduced to capture this deviation from randomness in temporal event sequences. It is defined based on the statistical properties of inter-event times.

For a sequence of inter-event times $ \{ \tau_1, \tau_2, \dots, \tau_n \} $, the burstiness index $ B $ is defined as:

$$ B = \frac{\sigma_\tau - \mu_\tau}{\sigma_\tau + \mu_\tau} $$

where:
- $ \mu_\tau $ is the mean of the inter-event times,
- $ \sigma_\tau $ is the standard deviation of the inter-event times.


The burstiness index $ B $ has the following properties:
- $ B = -1 $: Events are perfectly regular (e.g., periodic).
- $ B = 0 $: What a Poisson process gives ($ \sigma = \mu $). The converse doesn't hold, since $ B $ only sees $ \sigma/\mu $.
- $ B > 0 $: Events are bursty: more variable than a Poisson process. A heavy tail is one way to get there, not the only one, and a power law with a steep enough tail has $ B < 0 $.

#### Intuition Behind the Formula
The burstiness index is designed to compare the variability ($ \sigma $) of inter-event times to their mean ($ \mu $):
- When $ \sigma \approx \mu $, the inter-event times vary about as much as exponential gaps do (the Poisson case), and $ B $ approaches 0.
- When $ \sigma \gg \mu $, the variability dominates, indicating highly irregular event timing, and $ B $ becomes positive.
- When $ \sigma \ll \mu $, as in periodic systems, $ B $ becomes negative.

__________

#### Applications of Burstiness Index
The burstiness index has been widely used to study temporal networks in various contexts:
1. **Social Networks**: Understanding the temporal patterns of human communication (e.g., email, messaging, phone calls).
2. **Biological Systems**: Analyzing neuronal firing patterns or animal behavior.
3. **Infrastructure Networks**: Characterizing traffic patterns or network usage in telecommunication systems.
4. **Epidemiology**: Modeling the timing of human interactions that drive disease spread.

#### Importance in Temporal Networks
In temporal networks, burstiness provides critical insights into the structure and dynamics of event sequences. For example:
- It influences how fast information or diseases propagate.
- It reveals behavioral rhythms and anomalies in human activity.
- Together with the memory coefficient, it separates classes of system: Goh & Barabási place human activity, natural phenomena, heartbeats, and written texts in different regions.


The burstiness index is thus an essential tool for exploring and characterizing temporal networks, enabling researchers to identify deviations from randomness and uncover patterns underlying complex systems.

![](images/bursts_barabasi_nature2005.png)

![](images/srep00397-f1.jpg)

**References**
1. Barabási, AL. The origin of bursts and heavy tails in human dynamics. *Nature* 435, 207–211 (2005). https://doi.org/10.1038/nature03459
2. Goh, K. I., & Barabási, AL (2008). Burstiness and memory in complex systems. Europhysics Letters, 81(4), 48002. https://iopscience.iop.org/article/10.1209/0295-5075/81/48002
3. Karsai, M., Kaski, K., Barabási, AL. et al. Universal features of correlated bursty behaviour. *Sci Rep* 2, 397 (2012). https://doi.org/10.1038/srep00397

### Origin of burstiness? Queue Model

Possible cause? Choosing tasks by priority rather than at random.

#### Priority List Model
1. Pick the task with the highest priority with probability $ p $, or randomly pick one with $ 1-p $, and execute it.
2. Generate a new task with a random priority.

| **Task** | **Priority** |
|----------|--------------|
| 1        | $ x_1 $    |
| 2        | $ x_2 $    |
| 3        | $ x_3 $    |
| 4        | $ x_4 $    |
| 5        | $ x_5 $    |
| 6        | $ x_6 $    |
| ...      | ...        |
| $ L $  | $ x_L $    |

To get the tail analytically, Barabási swaps the $p$ rule for a stochastic version that matches it only in the two limits below. The probability of executing a task is $\Pi(x) = \dfrac{x^{\gamma}}{\sum_{i=1\cdots L} x_i^{\gamma}}$.
- $\gamma = 0$: random ($p=0$)
- $\gamma = \infty$: deterministic ($p=1$)

The average waiting time of a task with priority $x$ is $\tau(x) = \dfrac{1}{\Pi(x)} \sim x^{-\gamma}$.

In [ ]:
def measure_burstiness(inter_event_times):
    """
    Calculate the burstiness index for a sequence of inter-event times.

    The burstiness index is defined as:
        B = (sigma - mu) / (sigma + mu)
    where:
        - sigma is the standard deviation of the inter-event times.
        - mu is the mean of the inter-event times.

    Parameters:
    ----------
    inter_event_times : array-like
        A sequence of inter-event times (time differences between consecutive events).

    Returns:
    -------
    float
        The burstiness index, where:
        - B = -1 indicates perfectly regular events.
        - B = 0 indicates Poissonian/random events.
        - B > 0 indicates bursty events.
    
    Raises:
    ------
    ValueError
        If the input array is empty or has insufficient data to calculate burstiness.
    """
    # Ensure the input is a valid array
    if len(inter_event_times) < 2:
        raise ValueError("At least two inter-event times are required to calculate burstiness.")
    
    # Compute mean and standard deviation
    mu = np.mean(inter_event_times)
    sigma = np.std(inter_event_times)

    # Avoid division by zero
    if sigma + mu == 0:
        return 0.0

    # Calculate and return the burstiness index
    return (sigma - mu) / (sigma + mu)

In [ ]:
print("burstiness index = %.6f" % measure_burstiness(ie_time_list))
# Those gaps are pooled over 180 students and include nights and the weekend (the bumps in the figure above)
print("without gaps over 6 hours = %.6f" % measure_burstiness([t for t in ie_time_list if t <= 6 * 3600]))

In [ ]:
ts_length = 100000

# A Poisson process has exponentially distributed gaps between events, so sigma = mu and B = 0
random_time_list = np.random.exponential(size=ts_length)

periodic_time_list = [10] * ts_length

# Zipf(2) has infinite mean and variance, so this B grows with ts_length and changes with the seed.
# Try 4.0 in place of 2.0: still a heavy tail, but B < 0
powerlaw_time_list = np.random.zipf(2.0, ts_length)

print("Periodic inter-event times:\tBurstiness = %.3f"%measure_burstiness(periodic_time_list))
print("Random inter-event times:\tBurstiness =  %.3f"%measure_burstiness(random_time_list))
print("Powerlaw inter-event times:\tBurstiness =  %.3f"%measure_burstiness(powerlaw_time_list))


____________
## Activity-Driven Network Model (if there's time!)

The **Activity-Driven Network Model** is a widely used framework for studying dynamic networks, where the structure evolves over time based on the intrinsic activity levels of nodes. Unlike static networks, where connections remain fixed, activity-driven networks capture the temporal nature of interactions, making them suitable for analyzing systems such as social interactions, communication patterns, and information diffusion.

![](images/activity_driven.png)


### Key Features of the Model
The activity-driven network model assumes that each node in the network is characterized by an **activity rate** $a_i$, which determines the probability that the node becomes active in a given time step. When a node becomes active, it forms temporary edges with other nodes in the network. These edges exist only for the duration of the time step, reflecting the transient nature of interactions in many real-world systems.

At each time step:
1. **Node Activation**: Each node becomes active with a probability proportional to its activity rate $ a_i $.
2. **Edge Formation**: Active nodes form $m$ connections with other randomly selected nodes in the network.
3. **Temporal Graph**: The resulting network for the time step is recorded as a temporal graph.
4. **Aggregation**: Over multiple time steps, an aggregated network is constructed, representing all the edges formed throughout the simulation.

---

### Model Parameters
The activity-driven network model is controlled by the following parameters:
- $ N $: The number of nodes in the network.
- $ T $: The number of time steps over which the network evolves.
- $ m $: The number of edges each active node creates per time step.
- $ a_i $: The activity rate of node $ i $, sampled from a specified **activity distribution** (e.g., uniform or power-law).

The choice of the activity distribution is crucial, as it can reflect heterogeneous or homogeneous activity patterns. For example:
- A **narrow distribution** implies that all nodes have similar probabilities of becoming active.
- A **power-law distribution** assigns higher activity rates to a few nodes, reflecting real-world systems where a small subset of nodes drives most interactions.

#### Applications
The activity-driven network model is versatile and has been applied in various domains:
1. **Social Networks**: Simulating interactions in human systems, such as email exchanges or face-to-face contact.
2. **Epidemiology**: Modeling the spread of infectious diseases, where temporary connections between individuals facilitate disease transmission.
3. **Infrastructure**: Understanding communication or transportation networks, where connections appear dynamically over time.
4. **Information Diffusion**: Capturing how information propagates through dynamic networks.

By explicitly modeling the temporal dynamics of networks, this approach provides a powerful tool for studying processes that unfold over time, such as epidemic outbreaks, cascades, and synchronization phenomena.
_________ 



### Model dynamics:
There are $N$ nodes in the network, and each node has an activity potential $x$, extracted from a distribution $F(x)$.

1. At each time step $t$ the network $G_t$ starts with $N$ disconnected nodes
2. The node $i$ fires with probability $a_i \Delta t$, where $a_i = \eta x_i$ and $\eta$ is a rescaling factor
3. Each active node generates $m$ links connecting to $m$ other randomly selected nodes (could be either active or non-active)
4. At next time step $t+\Delta t$, all links in $G_t$ are deleted. Go to step 1.


**Notes:**
- $a_i = \eta x_i$ : Probability per unit time to create new connections with other nodes
- $\epsilon \leq x_i \leq 1 $
- The average number of active nodes per unit time is $\eta \langle x \rangle N$


**Therefore:**
- Total number of edges per unit time: $ E_t = mN\eta\langle x \rangle $

- The average degree per unit time: $ k_t = \frac{2E_t}{N} = 2m\eta\langle x \rangle $

- At time $T$, on the aggregated network up to time $T$, the degree of node $i$: $ k_i(T)=N(1-e^{-Tm\eta x_i/N}) $

- That formula counts the links node $i$ creates. Node $i$ also receives links from other active nodes, which adds $\langle a \rangle$ to its rate: $ k_i(T) \approx N(1-e^{-Tm(a_i+\langle a \rangle)/N}) $. So it's most accurate for the highly active nodes that make up the tail.

- So for $k \ll N$ and $k \ll T$, and in the tail where $a_i \gg \langle a \rangle$, the degree distribution for the aggregated network up to time $T$ (the simulation below, with $N=100$ and $T=50$, is outside this regime): $ P_T(k) \sim F\Big [ \frac{k}{Tm\eta} \Big ] $


**References**
1. Perra, N., Gonçalves, B., Pastor-Satorras, R. et al. Activity driven modeling of time varying networks. *Sci Rep* 2, 469 (2012). https://doi.org/10.1038/srep00469
2. Perra, N., Baronchelli, A., Mocanu, D., Gonçalves, B., Pastor-Satorras, R., & Vespignani, A. (2012). Random walks and search in time-varying networks. *Physical Review Letters*, 109(23), 238701. https://doi.org/10.1103/PhysRevLett.109.238701

In [ ]:
def activity_driven_network(N, T, m, activity_distribution):
    """
    Simulate an activity-driven network model.

    Parameters:
    ----------
    N : int
        Number of nodes in the network.
    T : int
        Number of time steps to simulate.
    m : int
        Number of edges each active node forms at each time step.
    activity_distribution : callable
        A function that generates activity rates for the nodes.
        Example: lambda: np.random.uniform(0, 1) for uniform distribution.

    Returns:
    -------
    G_agg : networkx.Graph
        The aggregated graph over all time steps.
    G_t_list : list of networkx.Graph
        A list of temporal graphs (one for each time step).
    """

    # Step 1: Initialize nodes with activity rates
    activity_rates = np.array([activity_distribution() for _ in range(N)])
    nodes = list(range(N))

    # Step 2: Create an empty aggregated graph
    G_agg = nx.Graph()
    G_agg.add_nodes_from(nodes)

    # Step 3: Initialize list to store temporal graphs
    G_t_list = []

    # Step 4: Simulate the network over T time steps
    for t in range(T):
        # Create a new graph for the current time step
        G_t = nx.Graph()
        G_t.add_nodes_from(nodes)

        # Determine active nodes
        active_nodes = [node for node in nodes if np.random.rand() < activity_rates[node]]

        # Create connections for each active node
        for node in active_nodes:
            # Choose m distinct other nodes to connect to
            others = [n for n in nodes if n != node]
            targets = np.random.choice(others, size=m, replace=False)
            for target in targets:
                G_t.add_edge(node, target)
                G_agg.add_edge(node, target)  # Update the aggregated graph

        # Store the temporal graph for the current time step
        G_t_list.append(G_t)


    return G_agg, G_t_list

In [ ]:
# Example parameters
N = 100  # Number of nodes
T = 50   # Number of time steps
m = 3    # Number of edges per active node
activity_distribution = lambda: np.random.uniform(0.1, 0.5)  # Uniform distribution of activity rates

# Simulate the activity-driven network
G_agg, G_t_list = activity_driven_network(N, T, m, activity_distribution)

# Print basic stats
print(f"Aggregated Network: {G_agg.number_of_nodes()} nodes, {G_agg.number_of_edges()} edges")
print(f"Temporal Graphs: {len(G_t_list)} time steps")



__________
## Next time...
Network Sampling `class_16_sampling.ipynb` (Mon, Nov 9)
_______

## References and further resources:

1. Class Webpages
    - Jupyter Book: https://computational-methods-network-science.github.io/nets7052_fa26/README.html
    - Github: https://github.com/computational-methods-network-science/nets7052_fa26/
    - Syllabus and course details: https://brennanklein.com/nets7052-fall26
2. Cattuto, C., Van den Broeck, W., Barrat, A., Colizza, V., Pinton, J. F., & Vespignani, A. (2010). Dynamics of person-to-person interactions from distributed RFID sensor networks. *PloS One*, 5(7), e11596. https://doi.org/10.1371/journal.pone.0011596
3. Starnini, M., Baronchelli, A., & Pastor-Satorras, R. (2013). Modeling human dynamics of face-to-face interaction networks. *Physical Review Letters*, 110(16), 168701. https://doi.org/10.1103/PhysRevLett.110.168701
4. Barabási, AL. The origin of bursts and heavy tails in human dynamics. *Nature* 435, 207–211 (2005). https://doi.org/10.1038/nature03459
5. Perra, N., Gonçalves, B., Pastor-Satorras, R. et al. Activity driven modeling of time varying networks. *Sci Rep* 2, 469 (2012). https://doi.org/10.1038/srep00469
6. Perra, N., Baronchelli, A., Mocanu, D., Gonçalves, B., Pastor-Satorras, R., & Vespignani, A. (2012). Random walks and search in time-varying networks. *Physical Review Letters*, 109(23), 238701. https://doi.org/10.1103/PhysRevLett.109.238701
7. Goh, K. I., & Barabási, A. L. (2008). Burstiness and memory in complex systems. Europhysics Letters, 81(4), 48002. https://iopscience.iop.org/article/10.1209/0295-5075/81/48002
8. Karsai, M., Kaski, K., Barabási, AL. et al. Universal features of correlated bursty behaviour. *Sci Rep* 2, 397 (2012). https://doi.org/10.1038/srep00397
9. Fournet, J., & Barrat, A. (2014). Contact patterns among high school students. *PLoS ONE*, 9(9), e107878. https://doi.org/10.1371/journal.pone.0107878
10. Holme, P., & Saramäki, J. (2012). Temporal networks. *Physics Reports*, 519(3), 97–125. https://doi.org/10.1016/j.physrep.2012.03.001
11. Masuda, N., & Lambiotte, R. (2016). *A Guide to Temporal Networks*. World Scientific. https://doi.org/10.1142/q0033
